# Deepfake Detector — Sequential Training (Free GPU on Colab)

This notebook trains the vision and audio deepfake detectors **dataset by dataset** on Google Colab's free T4 GPU (or Kaggle's P100/T4).

**Why sequential?** Indexing all 16+ datasets at once crashes Kaggle (OOM, truncated files, 9h timeout). This trainer:
1. Trains on one dataset at a time
2. Indexes the next dataset in a background thread while training
3. Saves a checkpoint after every dataset (crash-proof)
4. Carries model weights forward (continual learning)
5. Stops after processing 35+ GB of data

**Free GPU options:**
- Google Colab Free: T4 GPU, 12h sessions, ~78GB RAM
- Kaggle: T4x2 or P100, 30h/week GPU, 9h per session
- This notebook works on both

## Cell 1: Clone repo & install dependencies

In [ ]:
import os, sys

# Clone the repo (if not already cloned)
if not os.path.isdir('AI_Deepfake_Detector'):
    !git clone https://github.com/your-username/AI_Deepfake_Detector.git
    # Replace with your actual repo URL

os.chdir('AI_Deepfake_Detector')
sys.path.insert(0, os.getcwd())

# Install dependencies
!pip install -q torch torchvision timm tqdm librosa
!pip install -q kaggle  # for downloading datasets

# For audio (torchaudio may already be installed with torch)
try:
    import torchaudio
except ImportError:
    !pip install -q torchaudio

print('Setup complete!')
print(f'GPU available: {torch.cuda.is_available()}' if 'torch' in sys.modules else 'import torch first')
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

## Cell 2: Upload Kaggle API key

Go to https://www.kaggle.com/ → **Account** → **Create New API Token** → upload `kaggle.json` below.

In [ ]:
from google.colab import files
import shutil, os

# Upload kaggle.json
uploaded = files.upload()
kaggle_json = list(uploaded.keys())[0]

# Place it where the Kaggle CLI expects it
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.move(kaggle_json, os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

print('Kaggle API key configured!')

## Cell 3: Download all Kaggle datasets

This downloads all 18 datasets to `/content/data/`. Each dataset is downloaded sequentially. This step takes a while depending on dataset sizes and network speed.

In [ ]:
# Download all datasets — this will take a while!
# Datasets that are too large for Colab's ~100GB disk will be skipped.
!python -m backend.training.download_kaggle_datasets --data-root /content/data --modality both

## Cell 4: Check disk usage and downloaded datasets

In [ ]:
!df -h /content
!du -sh /content/data/* 2>/dev/null | sort -rh | head -20
print()
!ls -la /content/data/

## Cell 5: Build custom source list from downloaded data

The sequential trainer uses `--source` arguments to point at local paths instead of Kaggle mounts.

In [ ]:
import os

data_root = '/content/data'
available = sorted(os.listdir(data_root)) if os.path.isdir(data_root) else []

print(f'Available datasets in {data_root}:')
for d in available:
    full = os.path.join(data_root, d)
    if os.path.isdir(full):
        size_mb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fns in os.walk(full) for f in fns) / (1024*1024)
        print(f'  {d}: {size_mb:.0f} MB')

print(f'\nTotal datasets: {len(available)}')

## Cell 6: Run sequential VISION training

Trains on each dataset one at a time. Model weights carry forward. Checkpoint saved after each dataset.

In [ ]:
# Build --source args for all downloaded vision datasets
# The sequential trainer will train on each one in order
import os, subprocess, sys

data_root = '/content/data'
output_dir = '/content/checkpoints'
os.makedirs(output_dir, exist_ok=True)

# List of vision dataset folder names (match DEFAULT_VISION_SOURCES)
vision_folders = [
    'deep-fake-detection-dfd-entire-original-dataset',
    'deepfake-and-real-images',
    '140k-real-and-fake-faces',
    'deepfake-image-detection',
    'deep-fake-detection-cropped-dataset',
    'comprehensive-deepfake-detection-dataset',
    '1-million-fake-faces-7',
    '1-million-fake-faces',
    '1m-ai-generated-faces-128x128',
    '1-million-fake-faces-2',
    '1-million-fake-faces-6',
    'realai-video-dataset',
    'ms1m-arcface-dataset',
]

# Build source args from what's actually available
source_args = []
for folder in vision_folders:
    path = os.path.join(data_root, folder)
    if os.path.isdir(path):
        source_args.extend(['--source', f'{folder}={path}'])

if not source_args:
    print('WARNING: No vision datasets found! Run Cell 3 first.')
else:
    print(f'Found {len(source_args)//2} vision datasets to train on')
    print(f'Output: {output_dir}')

    # Run sequential vision training
    cmd = [
        sys.executable, '-m', 'backend.training.train_sequential_kaggle',
        '--modality', 'vision',
        '--output-dir', output_dir,
        '--arch', 'cnn',
        '--backbone', 'efficientnet_b4',
        '--pretrained',
        '--epochs-per-dataset', '1',
        '--batch-size', '32',
        '--amp',
        '--resume',
        '--save-every', '500',
        '--target-gb', '35',
        '--workers', '2',
    ] + source_args

    print(f'\nCommand: {" ".join(cmd[:10])}...')
    result = subprocess.run(cmd)
    print(f'\nVision training exit code: {result.returncode}')

## Cell 7: Run sequential AUDIO training

In [ ]:
audio_folders = [
    'audio-deepfake-detection-dataset',
    'deepfake-audio-dataset-fake-vs-real-speech',
]

source_args = []
for folder in audio_folders:
    path = os.path.join(data_root, folder)
    if os.path.isdir(path):
        source_args.extend(['--source', f'{folder}={path}'])

if not source_args:
    print('WARNING: No audio datasets found!')
else:
    print(f'Found {len(source_args)//2} audio datasets')
    cmd = [
        sys.executable, '-m', 'backend.training.train_sequential_kaggle',
        '--modality', 'audio',
        '--output-dir', output_dir,
        '--epochs-per-dataset', '2',
        '--batch-size', '64',
        '--amp',
        '--resume',
        '--save-every', '100',
        '--target-gb', '35',
        '--workers', '2',
    ] + source_args

    result = subprocess.run(cmd)
    print(f'\nAudio training exit code: {result.returncode}')

## Cell 8: Download checkpoints to local machine

In [ ]:
from google.colab import files
import os

ckpt_dir = '/content/checkpoints'
for fname in os.listdir(ckpt_dir):
    if fname.endswith('.pth'):
        fpath = os.path.join(ckpt_dir, fname)
        size_mb = os.path.getsize(fpath) / (1024*1024)
        print(f'Downloading {fname} ({size_mb:.1f} MB)...')
        files.download(fpath)

## Cell 9: (Optional) Resume if training was interrupted

Just re-run Cell 6 and Cell 7 — the `--resume` flag will pick up from `seq_checkpoint.pth` automatically.

---
## Alternative: Run on Kaggle directly

If you prefer Kaggle (T4x2 = 2 GPUs), attach all datasets in the Kaggle notebook UI, then run:

```python
!python -m backend.training.train_sequential_kaggle \
    --modality both \
    --output-dir /kaggle/working/checkpoints \
    --pretrained --amp --resume \
    --epochs-per-dataset 1 \
    --batch-size 32 \
    --target-gb 35
```

The sequential trainer uses the default Kaggle paths automatically — no `--source` needed.